# Modelling v2

## Design decisions

Architecture and evaluation strategy: 4-block CNN, AdamW + CosineAnnealingLR, early stopping, per-country tercile labels, LOOCV across six Balkan countries. Updated to **v4 GEE export** (11 channels):

| # | Channel | Notes |
|---|---------|-------|
| 0–3 | VIIRS 2013–2016 | temporal nightlight trend |
| 4 | Built-up fraction | continuous (0–1), GHSL |
| 5 | Population (log1p) | GHSL 2015 |
| 6 | Elevation | metres, SRTM |
| 7 | Slope | degrees, SRTM-derived |
| 8 | Infra proximity (log1p) | distance to nearest built-up cell |
| 9 | **SMOD code** | GHSL settlement classification (new in v4) |
| 10 | ~~VIIRS 2017~~ | **excluded** — source of labels |

> ⚠️ **SMOD flag:** ch9 (SMOD) is currently included as an input feature. Consider removing it (`INPUT_CHANNELS = [0,1,2,3,4,5,6,7,8]`) to test its marginal contribution, or reserve it as a post-hoc explanatory variable (e.g. stratify accuracy by SMOD class to understand *why* transfer succeeds or fails across countries).

Normalisation statistics are recomputed from scratch in each LOOCV fold using only the five training countries to prevent leakage.

## V3 baseline results (32×32, log-ratio target)

Key findings from the completed v3 run (9 channels, `max(viirs_2013, viirs_2017) ≥ 0.3` blank filter):

- **CNN consistently outperforms logistic regression** on 32×32 patches across all six LOOCV folds.
- Mean accuracy gain over LR: **~+4 pp** | Mean AUC gain: **~+5.8 pp** (macro OvR).
- 8×8 CNN matched LR exactly (both ~0.60 mean acc) — 4 km patches contain no exploitable spatial structure; 32×32 (16 km) is the correct scale for this task.
- Serbia dominates patch counts (~5–7× other countries) due to geography; class balance held via per-country tercile labelling.
- **Target variable:** log-ratio `log1p(viirs_2017) − log1p(viirs_2013)`. Departure-from-momentum rejected (homogeneous positive signal = common macro shock, no spatial pattern).
- **Filter:** `max(viirs_2013, viirs_2017) ≥ 0.3 nW/cm²/sr` — retains both declining areas (lit in 2013, dark in 2017) and expansion patches (dark in 2013, lit in 2017). Selection bias is defensible; VIIRS 2017 is not an input so there is no ML leakage.

V4 re-run (below) adds SMOD (ch9) and uses the same architecture and filter.

## Imports and constants

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
from pathlib import Path

PATCH_SIZE = 32
TARGET     = "logratio"   # "logratio" or "momentum"

# v4 patches — generated by eda_v2.ipynb from balkans_{country}_v4.tif
PATCH_DIR  = Path(f"data/patches_v4_{PATCH_SIZE}x{PATCH_SIZE}")

COUNTRIES = [
    "Albania",
    "Bosnia_and_Herzegovina",
    "Kosovo",
    "Montenegro",
    "North_Macedonia",
    "Serbia",
]

# v4 channels: ch0-8 same as v3, ch9=SMOD (new), ch10=viirs_2017 (excluded — target)
# NOTE: to test SMOD contribution, set INPUT_CHANNELS = [0,1,2,3,4,5,6,7,8]
INPUT_CHANNELS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # includes SMOD; excludes ch10 viirs_2017

CHANNEL_NAMES = [
    "VIIRS 2013", "VIIRS 2014", "VIIRS 2015", "VIIRS 2016",
    "Built-up fraction", "Population (log1p)",
    "Elevation", "Slope", "Infra proximity (log1p)", "SMOD code",
]

N_CLASSES = 3
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print(f"Input channels ({len(INPUT_CHANNELS)}):", [CHANNEL_NAMES[i] for i in INPUT_CHANNELS])


## Load patches

In [ ]:
all_patches = {}
all_labels  = {}

for country in COUNTRIES:
    patches = np.load(PATCH_DIR / f"{country}_patches.npy")[:, INPUT_CHANNELS, :, :].astype(np.float32)
    labels  = np.load(PATCH_DIR / f"{country}_labels_{TARGET}.npy").astype(np.int64)
    all_patches[country] = patches
    all_labels[country]  = labels
    print(f"{country}: {patches.shape} | class counts {np.bincount(labels)}")


## Dataset and DataLoaders

In [ ]:
def augment_patch(patch):
    if np.random.random() > 0.5:
        patch = np.flip(patch, axis=2).copy()
    if np.random.random() > 0.5:
        patch = np.flip(patch, axis=1).copy()
    k = np.random.randint(0, 4)
    if k > 0:
        patch = np.rot90(patch, k=k, axes=(1, 2)).copy()
    return patch


class PatchDataset(Dataset):
    def __init__(self, patches, labels, mean, std, augment=False):
        self.patches = patches
        self.labels  = labels
        self.augment = augment
        self.mean    = torch.tensor(mean, dtype=torch.float32).view(-1, 1, 1)
        self.std     = torch.tensor(std,  dtype=torch.float32).view(-1, 1, 1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        patch = self.patches[idx].copy()
        if self.augment:
            patch = augment_patch(patch)
        x = (torch.tensor(patch, dtype=torch.float32) - self.mean) / (self.std + 1e-8)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def make_loaders(test_country, batch_size=32, val_frac=0.2, seed=42):
    train_countries = [c for c in COUNTRIES if c != test_country]

    tr_patches = np.concatenate([all_patches[c] for c in train_countries])
    tr_labels  = np.concatenate([all_labels[c]  for c in train_countries])

    # normalisation computed on training countries only — no leakage
    mean = tr_patches.mean(axis=(0, 2, 3))
    std  = tr_patches.std(axis=(0, 2, 3))

    tr_p, val_p, tr_l, val_l = train_test_split(
        tr_patches, tr_labels, test_size=val_frac, random_state=seed, stratify=tr_labels
    )

    train_loader = DataLoader(
        PatchDataset(tr_p,  tr_l,  mean, std, augment=True),
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        PatchDataset(val_p, val_l, mean, std, augment=False),
        batch_size=batch_size, shuffle=False
    )
    test_loader = DataLoader(
        PatchDataset(all_patches[test_country], all_labels[test_country], mean, std, augment=False),
        batch_size=batch_size, shuffle=False
    )
    return train_loader, val_loader, test_loader


# sanity check
tr, val, te = make_loaders("Kosovo")
print(f"Train batches: {len(tr)} | Val batches: {len(val)} | Test batches: {len(te)}")


## CNN architecture

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(p=0.1),
        )

    def forward(self, x):
        return self.block(x)


class NightlightCNN(nn.Module):
    """
    4-block CNN for patch_size >= 32: 32->16->8->4->2, AdaptiveAvgPool->Linear.
    Matches Jean et al. (2016) spirit: learn spatial features from multi-channel
    satellite composites to predict economic outcomes.
    """
    def __init__(self, in_channels=10, n_classes=3, patch_size=32):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


# sanity check
dummy = torch.zeros(4, len(INPUT_CHANNELS), PATCH_SIZE, PATCH_SIZE)
model_test = NightlightCNN(in_channels=len(INPUT_CHANNELS), patch_size=PATCH_SIZE)
print(f"PATCH_SIZE={PATCH_SIZE} → output shape: {model_test(dummy).shape}")
n_params = sum(p.numel() for p in model_test.parameters())
print(f"Parameters: {n_params:,}")


## Training function

In [ ]:
def train_fold(test_country, epochs=80, lr=1e-3, patience=15):
    train_loader, val_loader, test_loader = make_loaders(test_country)

    model     = NightlightCNN(in_channels=len(INPUT_CHANNELS), n_classes=N_CLASSES,
                             patch_size=PATCH_SIZE).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss    = float("inf")
    patience_counter = 0
    best_state       = None
    history          = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(y)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                val_loss += criterion(model(X), y).item() * len(y)
        val_loss /= len(val_loader.dataset)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stop at epoch {epoch + 1}")
                break

    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_probs, all_true = [], [], []
    with torch.no_grad():
        for X, y in test_loader:
            logits = model(X.to(DEVICE))
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_true.append(y.numpy())

    y_true  = np.concatenate(all_true)
    y_pred  = np.concatenate(all_preds)
    y_probs = np.concatenate(all_probs)

    return {
        "country":  test_country,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1":       f1_score(y_true, y_pred, average="weighted"),
        "auc":      roc_auc_score(y_true, y_probs, multi_class="ovr", average="macro"),
        "cm":       confusion_matrix(y_true, y_pred),
        "history":  history,
        "n_test":   len(y_true),
        "y_true":   y_true,
        "y_pred":   y_pred,
        "y_probs":  y_probs,
    }


## LOOCV loop

In [ ]:
import pickle, os
results_path = f"data/loocv_results_v4_{PATCH_SIZE}x{PATCH_SIZE}_{TARGET}.pkl"

if os.path.exists(results_path):
    with open(results_path, "rb") as f:
        results = pickle.load(f)
    print("Loaded saved results.")
else:
    torch.manual_seed(42)
    np.random.seed(42)
    results = []
    for country in COUNTRIES:
        print(f"\n--- Fold: test on {country} ---")
        r = train_fold(country)
        results.append(r)
        print(f"  Accuracy: {r['accuracy']:.3f}  F1: {r['f1']:.3f}  AUC: {r['auc']:.3f}  (n={r['n_test']})")
    with open(results_path, "wb") as f:
        pickle.dump(results, f)
    print("Saved results.")


## Results summary

In [ ]:
rows = [{
    "Country":   r["country"].replace("_", " "),
    "N test":    r["n_test"],
    "Accuracy":  round(r["accuracy"], 3),
    "F1 (wtd)":  round(r["f1"],       3),
    "AUC (ovr)": round(r["auc"],      3),
} for r in results]

df = pd.DataFrame(rows)
mean_row = {
    "Country":   "Mean",
    "N test":    "—",
    "Accuracy":  round(df["Accuracy"].mean(),  3),
    "F1 (wtd)":  round(df["F1 (wtd)"].mean(),  3),
    "AUC (ovr)": round(df["AUC (ovr)"].mean(), 3),
}
df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)
print(df.to_string(index=False))


## Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Confusion matrices per LOOCV fold — v4", fontsize=14, fontweight="bold")
class_names = ["Low", "Med", "High"]

for ax, r in zip(axes.flat, results):
    cm      = r["cm"]
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(class_names)
    ax.set_yticks(range(3)); ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{r['country'].replace('_', ' ')} (acc={r['accuracy']:.2f})")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=11)

plt.tight_layout()
plt.show()


## Loss curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Training & validation loss per fold — v4", fontsize=14, fontweight="bold")

for ax, r in zip(axes.flat, results):
    ax.plot(r["history"]["train_loss"], label="train", color="steelblue")
    ax.plot(r["history"]["val_loss"],   label="val",   color="firebrick")
    ax.set_title(r["country"].replace("_", " "))
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## Logistic regression baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

baseline_results = []
for country in COUNTRIES:
    train_countries = [c for c in COUNTRIES if c != country]
    tr_p = np.concatenate([all_patches[c] for c in train_countries])
    tr_l = np.concatenate([all_labels[c]  for c in train_countries])
    te_p = all_patches[country]
    te_l = all_labels[country]

    X_train = tr_p.mean(axis=(2, 3))  # patch-mean features
    X_test  = te_p.mean(axis=(2, 3))
    scaler  = StandardScaler().fit(X_train)
    clf     = LogisticRegression(max_iter=1000, random_state=42).fit(
        scaler.transform(X_train), tr_l
    )
    y_pred  = clf.predict(scaler.transform(X_test))
    y_probs = clf.predict_proba(scaler.transform(X_test))
    baseline_results.append({
        "Country":     country.replace("_", " "),
        "LR Accuracy": round(accuracy_score(te_l, y_pred), 3),
        "LR AUC":      round(roc_auc_score(te_l, y_probs, multi_class="ovr", average="macro"), 3),
    })

lr_df  = pd.DataFrame(baseline_results)
cnn_df = pd.DataFrame([{
    "Country":      r["country"].replace("_", " "),
    "CNN Accuracy": round(r["accuracy"], 3),
    "CNN AUC":      round(r["auc"], 3),
} for r in results])

cmp = cnn_df.merge(lr_df, on="Country")
cmp["Acc gain"] = (cmp["CNN Accuracy"] - cmp["LR Accuracy"]).round(3)
cmp["AUC gain"] = (cmp["CNN AUC"]      - cmp["LR AUC"]).round(3)

mean_row = {
    "Country":      "Mean",
    "CNN Accuracy": cmp["CNN Accuracy"].mean().round(3),
    "CNN AUC":      cmp["CNN AUC"].mean().round(3),
    "LR Accuracy":  cmp["LR Accuracy"].mean().round(3),
    "LR AUC":       cmp["LR AUC"].mean().round(3),
    "Acc gain":     cmp["Acc gain"].mean().round(3),
    "AUC gain":     cmp["AUC gain"].mean().round(3),
}
cmp = pd.concat([cmp, pd.DataFrame([mean_row])], ignore_index=True)
print(cmp.to_string(index=False))


## CNN vs LR bar chart

In [ ]:
countries_short = [
    r["country"].replace("_", " ").replace("Bosnia and Herzegovina", "Bosnia")
    for r in results
]
cnn_accs = [r["accuracy"] for r in results]
lr_accs  = [
    float(cmp[cmp["Country"] == r["country"].replace("_", " ")]["LR Accuracy"].values[0])
    for r in results
]

x     = np.arange(len(countries_short))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, cnn_accs, width, label="CNN v2 (32×32)",     color="steelblue", alpha=0.85)
ax.bar(x + width/2, lr_accs,  width, label="Logistic Regression", color="orange",    alpha=0.85)
ax.axhline(1/3, color="black", linestyle="--", linewidth=1, label="Random baseline (33%)")
ax.set_xticks(x)
ax.set_xticklabels(countries_short, rotation=15, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("CNN v2 vs Logistic Regression — LOOCV accuracy per fold (v4 data)")
ax.set_ylim(0, 0.85)
ax.legend()
plt.tight_layout()
plt.show()


## Confidence distribution

In [ ]:
all_conf_correct = []
all_conf_wrong   = []

for r in results:
    conf = r["y_probs"].max(axis=1)
    all_conf_correct.extend(conf[r["y_true"] == r["y_pred"]].tolist())
    all_conf_wrong.extend(conf[r["y_true"] != r["y_pred"]].tolist())

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(all_conf_correct, bins=30, range=(0.3, 1.0), alpha=0.7,
        color="steelblue", label=f"Correct (n={len(all_conf_correct)})")
ax.hist(all_conf_wrong,   bins=30, range=(0.3, 1.0), alpha=0.7,
        color="firebrick", label=f"Wrong (n={len(all_conf_wrong)})")
ax.axvline(1/3, color="black", linestyle="--", linewidth=1, label="Random (0.33)")
ax.set_xlabel("Max softmax probability")
ax.set_ylabel("Patch count")
ax.set_title("Prediction confidence — correct vs wrong (all folds)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean confidence — correct: {np.mean(all_conf_correct):.3f} | wrong: {np.mean(all_conf_wrong):.3f}")
